# VAE + Semi-NMF on MNIST

The main experiment from my thesis. The VAE learns a representation and the matrix factorization learns cluster assignments at the same time.

$\mu(X) \approx YC$, where $Y\in[0,1]^{N\times k}$ and $C\in\mathbb{R}^{k\times d}$.

The definitions are in `functions.ipynb`. Start here, change the variables below, then run all cells. The short defaults are for trying the code, not reproducing the thesis table.


In [ ]:
%run ./functions.ipynb


## Data and parameters


In [ ]:
dataset = 'mnist'                 # mnist, fashion_mnist, digits, reuters10k
n_samples = 2000                  # None uses all available samples
split = 'all'                     # train + test, for transductive clustering
seed = 42
n_clusters = 10
hidden_dims = [128, 64]
latent_dim = 10
pretrain_epochs = 5
epochs = 20                       # joint epochs, after pretraining
batch_size = 256
lr = 1e-3
warm_epochs = 50
kappa = 1e-4
gamma = 1e-3
architecture = 'fc'               # 'conv' for MNIST / Fashion-MNIST
binary_mode = 'improved'          # none, primp, improved, legacy, legacy_alt
exclusive_mode = 'thesis'         # none, thesis, exact
checkpoint = None                # e.g. 'results/mnist_checkpoint.pt'
resume = None                    # same path to continue an interrupted run

# Larger experiment using the thesis architecture:
# n_samples = None
# hidden_dims = [500, 500, 2000]
# pretrain_epochs = 100
# epochs = 500
# This is an explicit training budget; historical run settings were not all saved.

seed_everything(seed)
X, y = load_data(dataset, n_samples=n_samples, seed=seed, split=split)
dimensions = [X.shape[1], *hidden_dims, latent_dim]
print(dataset, X.shape, 'device:', device)


## Train

Pretraining uses reconstruction and KL loss. Joint training then alternates two Adam updates: `(VAE, C)` with `Y` detached, followed by `(VAE, Y)` with `C` detached. The forward pass is recomputed between the two updates. Clustering uses the posterior mean; decoding uses a sampled latent vector.

The reconstruction and KL terms are summed over dimensions and averaged over samples. The MF term is averaged over all matrix entries. `kappa` and `gamma` are mapping strengths applied after the Adam step.

`exclusive_mode='thesis'` keeps the shrinkage printed in Eq. 65. `'exact'` uses the exact squared-L1 proximal threshold. The adaptive PRIMP update encourages near-one-hot rows; it does not enforce them exactly.


In [ ]:
model = None
if architecture == 'conv':
    seed_everything(seed)
    model = ConvolutionalVAE(input_dim=X.shape[1], latent_dim=latent_dim).to(device)

result = train_vae_mf_model_optimized(
    X, n_clusters=n_clusters, dimensions=dimensions, model=model,
    pretrain_epochs=pretrain_epochs, epochs=epochs,
    batch_size=batch_size, lr=lr, warm_epochs=warm_epochs,
    beta_start=0.1, beta_end=1.0, mf_weight=1.0,
    kappa=kappa, gamma=gamma, double_every=50,
    binary_mode=binary_mode, exclusive_mode=exclusive_mode,
    loss='mse' if dataset == 'reuters10k' else 'bce',
    seed=seed, y=y, checkpoint=checkpoint, resume=resume,
)
model, mf = result['model'], result['mf']
Z, pred, history = result['Z'], result['pred'], result['history']
pd.DataFrame([score_dict(y, pred)], index=[dataset])


## Check the run


In [ ]:
pretrain_history = result.get('pretrain_history', result.get('prehistory', {}))
full_history = {}
for key in ['loss', 'recon_loss', 'kl_loss', 'mf_loss']:
    before = pretrain_history.get(key, [0.] * len(pretrain_history.get('loss', [])))
    full_history[key] = before + history[key]
plot_history(full_history)
metric_names = ['ACC', 'NMI', 'ARI', 'F1', 'Purity']
metric_history = pd.DataFrame({key: history[key] for key in metric_names if key in history})
if len(metric_history):
    metric_history.plot(figsize=(8, 4))
    plt.xlabel('Joint epoch (after pretraining)')
    plt.ylabel('Score')
    plt.tight_layout()
    plt.show()
print(assignment_diagnostics(mf.Y.detach()))


In [ ]:
# First few assignment rows. These are continuous values, not forced one-hot labels.
plt.figure(figsize=(8, 3))
plt.imshow(mf.Y[:40].detach().cpu().numpy(), aspect='auto', vmin=0, vmax=1)
plt.xlabel('Cluster')
plt.ylabel('Sample')
plt.colorbar(label='Y')
plt.tight_layout()
plt.show()


## Save results if needed

The notebooks do not need old trained weights. A new run creates its own results. `checkpoint` above saves model, optimizer and random states for resuming; the file below is a small export for inspecting the final embedding and assignments.


In [ ]:
save_results = False
if save_results:
    output = Path('results') / dataset
    output.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(output / 'embedding.npz', Z=Z, y=y, pred=pred,
                        Y=mf.Y.detach().cpu().numpy(), C=mf.C.detach().cpu().numpy())
    pd.DataFrame(history).to_csv(output / 'history.csv', index=False)
    pd.DataFrame([score_dict(y, pred)]).to_csv(output / 'metrics.csv', index=False)


The earlier notebooks also tried delayed binary updates, cyclic MF weights and periodic center initialization. These remain optional arguments of `train_vae_mf_model_optimized`: `prox_start_epoch`, `mf_cycle` and `reinitialize_every`. `restart_kl=True` also restores the old two-phase KL warm-up. They are off in this run. The accumulated updates are available through `binary_mode='legacy'` and `'legacy_alt'`.

For two-stage models, open `baselines.ipynb`. For cluster-count experiments, loss ablations, latent plots and confusion maps, open `analysis.ipynb`.
